# real-chart-bench: LineFormer pretrained baseline (Google Colab)

## 使い方 (Usage)
**`Runtime` → `Run all`(ランタイム → すべてのセルを実行)を1回押すだけです。**
事前準備は `Runtime → Change runtime type → T4 GPU`(または上位)を選ぶことだけ。
完走すると結果が `/content/lineformer-pretrained.json` に自動保存され、ブラウザへも
自動ダウンロードされます — 追加の操作は不要です。所要時間の目安は5〜20分
(mmcv/mmdetectionのインストールが大半を占め、推論自体はすぐ終わります)。

Runs the LineFormer pretrained model (ICDAR2023, arxiv 2305.01837) against the
**real-chart-bench** v0 verified-image evaluation suite.

**Why Colab, not local**: `mmcv` ships source-only on PyPI and OpenMMLab's
prebuilt wheels are Linux+CUDA only — exactly what a Colab GPU runtime
provides for free (see `docs/design/benchmark-architecture.md` §7.16 for the
local-infeasibility writeup).

**Design history**: this notebook has been rebuilt across many Colab runs to
remove every point of manual intervention — an isolated Python 3.10 env for
LineFormer (no kernel restart ever needed), full-traceback error propagation,
an all-figures-failed guard, etc. This notebook itself now only carries the
short version of each fix; full root-cause writeups live in
`docs/design/benchmark-architecture.md` §7.35–§7.40.

**Scope note**: real-image evaluation is gated on `data/verified_pairs/registry.json`
(§7.19: "量より信頼性" — reliability over quantity). Only `status: "verified"`
entries are used; this is intentionally a small, trustworthy set rather than
the full (unverified) image↔ground-truth pairing.

In [ ]:
# ============================================================
# Setup: clone real-chart-bench, install it, build LineFormer's isolated
# Python 3.10 environment (torch/mmcv-full/mmdetection), download the
# pretrained checkpoint. One cell, no manual steps in between.
# Full history of why each piece is shaped this way:
# docs/design/benchmark-architecture.md §7.19/§7.26/§7.35-§7.37.
# ============================================================

# --- 1. Clone + install real_chart_bench (regular install, not -e:
# editable installs don't reliably import in the same kernel session
# without a restart, design §7.26) ---
!rm -rf /content/real-chart-bench
!git clone --depth 1 https://github.com/t29mato/real-chart-bench.git /content/real-chart-bench
%cd /content/real-chart-bench
%pip install -q .
%pip install -q pymupdf requests pillow gdown uv

import importlib

importlib.invalidate_caches()
import real_chart_bench  # noqa: F401

print(f"OK: real_chart_bench installed at {real_chart_bench.__file__}")

# --- 2. LineFormer's own isolated Python 3.10 env, built with uv.
# Colab's default Python (3.12) has no torch==1.13.1 wheel at all (design
# §7.35); this notebook's own kernel never imports torch/mmcv/mmdet --
# LineFormerModelRunner (below) runs LineFormer in a subprocess under this
# env instead, so no kernel restart is ever needed. ---
!uv python install 3.10
PY310_VENV = "/content/lineformer_venv"
!uv venv --python 3.10 -q {PY310_VENV}
PY310 = f"{PY310_VENV}/bin/python"

# setuptools<81: several legacy OpenMMLab packages import the now-removed
# pkg_resources at runtime, and `uv venv` doesn't seed setuptools by
# default (design §7.37).
!uv pip install -q --python {PY310} "setuptools<81"

# torch==1.13.1+cu117: the exact combo LineFormer's install.sh was
# authored against; mmcv-full's wheel is torch-version-specific.
!uv pip install -q --python {PY310} torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117

# mmcv-full (mmcv 1.x), NOT `mmcv` (mmcv 2.x) -- installed straight from
# OpenMMLab's own wheel index, not via `mim` (which crashes on
# pkg_resources in a bare uv venv, design §7.37).
!uv pip install -q --python {PY310} mmcv-full==1.7.2 \
    -f https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html

!git clone --depth 1 https://github.com/TheJaeLal/LineFormer.git /content/LineFormer

# LineFormer's own vendored mmdetection/ (not PyPI mmdet -- that pulls in
# the incompatible mmdet 3.x line). --no-build-isolation: its setup.py
# imports torch at module level, which a fresh isolated build env (uv's
# default) can't see (design §7.36).
!uv pip install -q --python {PY310} --no-build-isolation -e /content/LineFormer/mmdetection

!uv pip install -q --python {PY310} chardet scikit-image matplotlib \
    opencv-python pillow scipy==1.9.3 bresenham tqdm

# --- 3. Verify the install immediately, with full-stderr diagnostics on
# failure (design §7.37) -- and MPLBACKEND=Agg, since Colab's kernel env
# otherwise leaks an IPython-only backend name into this subprocess
# (design §7.36). ---
import os
import subprocess

_WORKER_ENV = {**os.environ, "MPLBACKEND": "Agg"}


def _check(label: str, code: str) -> bool:
    result = subprocess.run(
        [PY310, "-c", code],
        cwd="/content/LineFormer",
        capture_output=True,
        text=True,
        env=_WORKER_ENV,
    )
    if result.returncode == 0:
        print(f"OK: {label} (version={result.stdout.strip()})")
        return True
    stderr = result.stderr.strip() or "(no stderr captured)"
    print(f"FAILED: {label}\n--- stderr (full) ---\n{stderr}\n--- end stderr ---")
    return False


_checks_ok = True
_checks_ok &= _check("torch", "import torch; print(torch.__version__)")
_checks_ok &= _check("mmcv (mmcv-full)", "import mmcv; print(mmcv.__version__)")
_checks_ok &= _check(
    "mmdet (LineFormer's vendored mmdetection/)", "import mmdet; print(mmdet.__version__)"
)
_checks_ok &= _check(
    "infer (LineFormer's own module)",
    "import sys; sys.path.insert(0, '/content/LineFormer'); import infer; print('ok')",
)

if not _checks_ok:
    raise RuntimeError(
        "LineFormer stack install verification FAILED -- see the FAILED "
        "line(s) and full stderr above for which module and why. Common "
        "causes: Colab's GPU driver/CUDA build has drifted from the pinned "
        "torch==1.13.1+cu117 above (mmcv-full's wheel is torch-version-"
        "specific), or the mmdetection build_editable step above needs "
        "--no-build-isolation re-checked if its own error output mentions "
        "a missing build-time import."
    )
print("\nAll LineFormer/mmcv install checks passed (Python 3.10 subprocess env).")

# --- 4. Pretrained checkpoint. LineFormer's README only offers a Google
# Drive *folder* link (no direct-file URL) -- gdown --folder handles the
# large-file virus-scan-warning page automatically. ---
CHECKPOINT_DIR = "/content/LineFormer/checkpoints"
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/iter_3000.pth"
CONFIG_PATH = "/content/LineFormer/lineformer_swin_t_config.py"
CHECKPOINT_FOLDER_URL = (
    "https://drive.google.com/drive/folders/1K_zLZwgoUIAJtfjwfCU5Nv33k17R0O5T"
)

import pathlib

pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
if not pathlib.Path(CHECKPOINT_PATH).exists():
    !gdown --folder -q --continue -O {CHECKPOINT_DIR} "{CHECKPOINT_FOLDER_URL}"
    # gdown --folder preserves the Drive folder's own subfolder layout, so
    # the checkpoint may land one level deeper than CHECKPOINT_PATH -- find
    # it by filename and move it into place rather than assuming the path.
    if not pathlib.Path(CHECKPOINT_PATH).exists():
        found = list(pathlib.Path(CHECKPOINT_DIR).rglob("iter_3000.pth"))
        if found:
            found[0].rename(CHECKPOINT_PATH)

if pathlib.Path(CHECKPOINT_PATH).exists():
    size_mb = pathlib.Path(CHECKPOINT_PATH).stat().st_size / 1e6
    print(f"OK: checkpoint present at {CHECKPOINT_PATH} ({size_mb:.1f} MB)")
else:
    raise RuntimeError(
        f"Checkpoint download failed -- {CHECKPOINT_PATH} does not exist "
        f"after gdown. The Drive folder link may have moved; check "
        f"https://github.com/TheJaeLal/LineFormer#pretrained-models for the "
        f"current link and update CHECKPOINT_FOLDER_URL above."
    )

In [ ]:
# ============================================================
# Load verified-pair images + build the evaluation dataset.
# Bundled-first (design §7.33): every VERIFIED registry image is already
# committed under data/verified_pairs/images|crops/, no network needed. A
# bare-filename entry falls back to a live PDF re-fetch (skip-and-report on
# failure -- one bad fetch must not kill the whole run).
# ============================================================
%cd /content/real-chart-bench
import json
import pathlib

from real_chart_bench.adapter.verified_pairing_registry import load_registry
from real_chart_bench.usecase.real_image_gate import select_verified_pairings

REPO_ROOT = pathlib.Path("/content/real-chart-bench")
registry = load_registry(REPO_ROOT / "data/verified_pairs/registry.json")
verified = select_verified_pairings(registry)
print(f"{len(verified)} VERIFIED pairing(s) will be evaluated.")

# papers.json has no pdf_url (small committed metadata) -- only needed for
# the live-refetch fallback path below.
import urllib.parse
import urllib.request

from real_chart_bench.adapter.pdf_fetch import HttpPdfFetchAdapter
from real_chart_bench.adapter.figure_extraction import PyMuPdfFigureExtractor
from real_chart_bench.usecase.pdf_fetch import PdfFetchStatus

papers = json.loads((REPO_ROOT / "data/manifest/v0/papers.json").read_text())
papers_by_id = {p["paper_id"]: p for p in papers}


def resolve_pdf_url(doi: str) -> str | None:
    params = urllib.parse.urlencode({"filter": f"doi:{doi}", "per-page": 1})
    req = urllib.request.Request(
        f"https://api.openalex.org/works?{params}",
        headers={"User-Agent": "real-chart-bench/0.0.1 (mailto:tomoya.matou@gmail.com)"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.load(resp)
    results = data.get("results") or []
    if not results:
        return None
    work = results[0]
    best_oa = (work.get("best_oa_location") or {})
    primary = (work.get("primary_location") or {})
    return best_oa.get("pdf_url") or primary.get("pdf_url")


def resolve_image_path(pairing) -> pathlib.Path:
    # '/' in image_path = a committed repo-relative asset (design §7.21/§7.33).
    if "/" in pairing.image_path:
        return REPO_ROOT / pairing.image_path
    return REPO_ROOT / "data/raw/images" / pairing.paper_id / pairing.image_path


pdf_fetcher = HttpPdfFetchAdapter()
extractor = PyMuPdfFigureExtractor()

images_by_pairing: dict[tuple[str, str], bytes] = {}
skipped: list[dict] = []

for pairing in verified:
    key = (pairing.paper_id, pairing.figure_id)
    bundled_path = resolve_image_path(pairing)
    if bundled_path.exists():
        images_by_pairing[key] = bundled_path.read_bytes()
        continue
    try:
        paper = papers_by_id[pairing.paper_id]
        pdf_url = resolve_pdf_url(paper["doi"])
        if not pdf_url:
            raise RuntimeError(f"no pdf_url resolvable (doi={paper['doi']})")
        fetch_result = pdf_fetcher.fetch(pdf_url)
        if fetch_result.status is not PdfFetchStatus.OK or not fetch_result.content:
            raise RuntimeError(f"PDF fetch failed: {fetch_result.status}")
        extracted = extractor.extract(fetch_result.content)
        named = {}
        for i, img in enumerate(extracted):
            ext = "png" if img.source.value == "page_render" else "jpg"
            named[f"p{img.page_number:02d}_{img.source.value}_{i}.{ext}"] = img.image_bytes
        if pairing.image_path not in named:
            raise RuntimeError(
                f"expected image {pairing.image_path!r} not found among "
                f"{len(named)} re-extracted images"
            )
        images_by_pairing[key] = named[pairing.image_path]
    except Exception as exc:  # noqa: BLE001 -- one bad fetch must not kill the whole run
        skipped.append({"paper_id": pairing.paper_id, "figure_id": pairing.figure_id, "reason": str(exc)})
        print(f"SKIPPED paper {pairing.paper_id} figure {pairing.figure_id}: {exc}")

print(f"{len(images_by_pairing)} image(s) available, {len(skipped)} pairing(s) skipped")

# --- Build DatasetItems (real verified pairs + 3 synthetic fixtures),
# mirroring scripts/eval/run_baselines.py so results are comparable to the
# naive-CV baseline on the same figures. ---
from real_chart_bench.adapter.panel_layout import PyMuPdfPanelSplitter
from real_chart_bench.domain.curve import Curve, ScaleType
from real_chart_bench.usecase.evaluate_dataset import DatasetItem
from real_chart_bench.usecase.model_runner import ExtractionTask

# ground_truth.json is the committed subset of curve x/y values actually
# needed (data/manifest/v0/curves.json has metadata only; the full values
# live in a gitignored cache, design §7.33).
ground_truth_by_figure = json.loads((REPO_ROOT / "data/verified_pairs/ground_truth.json").read_text())


def ground_truth_for(figure_id: str) -> list[Curve]:
    return [
        Curve(x_values=tuple(r["x"]), y_values=tuple(r["y"]), series_label=r["prop_y"])
        for r in ground_truth_by_figure.get(figure_id, [])
        if r["x"]  # some Starrydata rows are empty digitization artifacts
    ]


real_items = []
splitter = PyMuPdfPanelSplitter()
for pairing in verified:
    key = (pairing.paper_id, pairing.figure_id)
    if key not in images_by_pairing:
        continue  # skipped above -- excluded from scoring, not a crash
    image_bytes = images_by_pairing[key]
    if pairing.panel_label is not None:
        panels = {p.label: p for p in splitter.split(image_bytes)}
        image_bytes = panels[pairing.panel_label].image_bytes
    task = ExtractionTask(
        image_bytes=image_bytes,
        x_range=pairing.x_range,
        y_range=pairing.y_range,
        x_scale=pairing.x_scale,
        y_scale=pairing.y_scale,
    )
    real_items.append(
        DatasetItem(
            figure_id=f"{pairing.paper_id}-{pairing.figure_id}",
            task=task,
            ground_truth=ground_truth_for(pairing.figure_id),
        )
    )


import pymupdf


def synthetic_items() -> list[DatasetItem]:
    items = []

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(1, 0, 0), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-linear-single",
        task=ExtractionTask(image_bytes=png, x_range=(0, 10), y_range=(0, 10)),
        ground_truth=[Curve(x_values=(0.0, 10.0), y_values=(10.0, 0.0))],
    ))

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 280), pymupdf.Point(280, 20), color=(1, 0, 0), width=2)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(0, 0, 1), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-linear-two-series",
        task=ExtractionTask(image_bytes=png, x_range=(0, 10), y_range=(0, 10)),
        ground_truth=[
            Curve(x_values=(0.0, 10.0), y_values=(0.0, 10.0), series_label="up"),
            Curve(x_values=(0.0, 10.0), y_values=(10.0, 0.0), series_label="down"),
        ],
    ))

    doc = pymupdf.open(); page = doc.new_page(width=300, height=300)
    page.draw_line(pymupdf.Point(20, 20), pymupdf.Point(280, 280), color=(0, 0, 0), width=2)
    png = page.get_pixmap().tobytes("png"); doc.close()
    items.append(DatasetItem(
        figure_id="synthetic-log-black-line",
        task=ExtractionTask(image_bytes=png, x_range=(1, 100), y_range=(0, 10), x_scale=ScaleType.LOG),
        ground_truth=[Curve(x_values=(1.0, 100.0), y_values=(10.0, 0.0), x_scale=ScaleType.LOG)],
    ))

    return items


dataset_items = real_items + synthetic_items()
print(
    f"{len(dataset_items)} total DatasetItem(s) "
    f"({len(real_items)} real + {len(dataset_items) - len(real_items)} synthetic, "
    f"{len(skipped)} skipped)"
)

In [ ]:
# ============================================================
# LineFormerModelRunner: wraps LineFormer's pretrained model behind the
# project's ModelRunnerPort protocol (same interface as NaiveCvModelRunner).
# Runs inference via SUBPROCESS in the isolated Python 3.10 env (PY310) --
# this notebook's own kernel never imports torch/mmcv/mmdet (design §7.35).
# Reloads the model per figure (slower, but one figure's failure can never
# corrupt another's -- a deliberate simplicity/reliability tradeoff).
# ============================================================
%pip install -q pillow

import io
import json
import os
import pathlib
import subprocess
import tempfile

from PIL import Image

from real_chart_bench.domain.curve import Curve
from real_chart_bench.usecase.model_runner import ExtractionTask

WORKER_SCRIPT_PATH = "/content/lineformer_infer_worker.py"

_WORKER_SCRIPT_SOURCE = '''\
"""LineFormer inference worker -- runs under the isolated Python 3.10 env,
one process per figure. Reads one image, writes one JSON result file.
"""
import argparse
import json
import sys
import traceback

sys.path.insert(0, "/content/LineFormer")

import cv2  # noqa: E402
import torch  # noqa: E402
import infer as lineformer_infer  # noqa: E402


def run(args) -> None:
    device = args.device
    if device == "auto":
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

    img = cv2.imread(args.image)
    if img is None:
        raise RuntimeError(f"cv2.imread returned None for {args.image!r}")

    lineformer_infer.load_model(args.config, args.checkpoint, device)

    # to_clean=False (matches LineFormer's own README example exactly):
    # to_clean=True requires a PMC-competition-format `annot` dict this
    # notebook never has, and crashes with a bare TypeError otherwise
    # (design §7.37, Colab run #11).
    series_list = lineformer_infer.get_dataseries(img, to_clean=False)

    # Each point is an {"x":, "y":} dict, not an (x, y) tuple (design §7.37).
    payload = [[[float(pt["x"]), float(pt["y"])] for pt in points] for points in series_list]
    with open(args.output, "w") as f:
        json.dump(payload, f)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--image", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    try:
        run(args)
    except Exception:
        print("=== LineFormer worker traceback (full) ===", file=sys.stderr)
        traceback.print_exc(file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()
'''

pathlib.Path(WORKER_SCRIPT_PATH).write_text(_WORKER_SCRIPT_SOURCE)


class LineFormerModelRunner:
    """ModelRunnerPort implementation that shells out to PY310 per figure."""

    def __init__(
        self,
        config_path: str,
        checkpoint_path: str,
        python_path: str,
        worker_script_path: str = WORKER_SCRIPT_PATH,
        device: str = "auto",
    ):
        self._config_path = config_path
        self._checkpoint_path = checkpoint_path
        self._python_path = python_path
        self._worker_script_path = worker_script_path
        self._device = device
        # MPLBACKEND=Agg: Colab's kernel env otherwise leaks an IPython-only
        # backend name into this subprocess (design §7.36).
        self._worker_env = {**os.environ, "MPLBACKEND": "Agg"}

    @staticmethod
    def _scale_frac(frac: float, lo: float, hi: float, is_log: bool) -> float:
        # Mirrors domain/pixel_calibration.py's PixelCalibration._scale_frac.
        if is_log:
            return lo * (hi / lo) ** frac
        return lo + frac * (hi - lo)

    def extract(self, task: ExtractionTask) -> list[Curve]:
        width, height = Image.open(io.BytesIO(task.image_bytes)).size

        with tempfile.TemporaryDirectory() as tmpdir:
            image_path = pathlib.Path(tmpdir) / "input.png"
            output_path = pathlib.Path(tmpdir) / "output.json"
            image_path.write_bytes(task.image_bytes)

            result = subprocess.run(
                [
                    self._python_path,
                    self._worker_script_path,
                    "--config", self._config_path,
                    "--checkpoint", self._checkpoint_path,
                    "--device", self._device,
                    "--image", str(image_path),
                    "--output", str(output_path),
                ],
                cwd="/content/LineFormer",
                capture_output=True,
                text=True,
                timeout=600,
                env=self._worker_env,
            )
            if result.returncode != 0:
                # Full stderr, not just the last line (design §7.37) -- this
                # ends up verbatim in the results JSON's per-figure "error"
                # field via evaluate_dataset.py.
                stderr = result.stderr.strip() or "(no stderr captured)"
                raise RuntimeError(
                    f"LineFormer worker subprocess failed (full stderr below):\n{stderr}"
                )

            series_list = json.loads(output_path.read_text())

        x0, x1 = task.x_range
        y0, y1 = task.y_range
        x_is_log = task.x_scale.name == "LOG"
        y_is_log = task.y_scale.name == "LOG"
        curves = []
        for i, points in enumerate(series_list):
            xs, ys = [], []
            for px, py in points:
                frac_x = px / width
                frac_y = 1.0 - (py / height)  # image y grows downward
                xs.append(self._scale_frac(frac_x, x0, x1, x_is_log))
                ys.append(self._scale_frac(frac_y, y0, y1, y_is_log))
            order = sorted(range(len(xs)), key=lambda j: xs[j])
            xs = [xs[j] for j in order]
            ys = [ys[j] for j in order]
            curves.append(Curve(x_values=tuple(xs), y_values=tuple(ys), series_label=f"series_{i}", x_scale=task.x_scale))
        return curves

In [ ]:
# ============================================================
# Run evaluation, write results/lineformer-pretrained.json, auto-download.
# This is the last cell -- when it finishes, you're done.
# ============================================================
from datetime import UTC, datetime

from real_chart_bench.domain.matching import HungarianCurveMatcher
from real_chart_bench.domain.metrics import NormalizedYDistanceMetric
from real_chart_bench.usecase.evaluate_dataset import evaluate_model_on_dataset

model = LineFormerModelRunner(CONFIG_PATH, CHECKPOINT_PATH, PY310)  # device auto-detected inside the worker subprocess
matcher = HungarianCurveMatcher(metric=NormalizedYDistanceMetric())
results = evaluate_model_on_dataset(model, dataset_items, matcher=matcher)

per_figure = [
    {
        "figure_id": r.figure_id,
        "summary_score": r.evaluation.summary_score,
        "match_rate": r.evaluation.match_rate,
        "mean_curve_distance": r.evaluation.mean_curve_distance,
        "mean_coverage_ratio": r.evaluation.mean_coverage_ratio,
        "error": r.error,
    }
    for r in results
]

# Refuse to write a results file if EVERY figure errored out (design §7.37):
# a 100%-failure run would otherwise produce a real-looking
# mean_summary_score=0.0 that could land on the leaderboard by mistake.
n_errors = sum(1 for p in per_figure if p["error"] is not None)
if per_figure and n_errors == len(per_figure):
    raise RuntimeError(
        f"All {len(per_figure)} figure(s) failed -- refusing to write a "
        f"results JSON with mean_summary_score=0.0. First figure's full "
        f"error ({per_figure[0]['figure_id']}):\n{per_figure[0]['error']}"
    )

mean_score = sum(p["summary_score"] for p in per_figure) / len(per_figure)

# dataset_version is derived from the actual evaluated pairing count, never
# hardcoded (design §7.28/§7.33).
payload = {
    "model_id": "lineformer-pretrained",
    "model_name": "LineFormer (pretrained, ICDAR2023)",
    "dataset_version": f"v0-eval-pilot-n{len(real_items)}",
    "run_at": datetime.now(UTC).isoformat(),
    "n_figures": len(per_figure),
    "mean_summary_score": mean_score,
    "n_verified_pairs_evaluated": len(real_items),
    "n_verified_pairs_skipped": len(skipped),
    "skipped_pairs": skipped,
    "per_figure": per_figure,
}

out_path = pathlib.Path("/content/lineformer-pretrained.json")
out_path.write_text(json.dumps(payload, indent=2))

print("=" * 70)
print(
    f"SUMMARY: evaluated {len(real_items)} verified real pairing(s) + "
    f"{len(dataset_items) - len(real_items)} synthetic fixture(s) = "
    f"{len(per_figure)} figure(s) scored ({n_errors} errored)."
)
print(f"mean_summary_score = {mean_score:.4f}")
print(f"Result written to: {out_path}")
print("=" * 70)
print(json.dumps(payload, indent=2))

# Auto-download to your computer -- no extra click needed (design §7.40,
# HQ instruction 2026-08-28: "開いて全実行して結果JSONを落とすだけ").
from google.colab import files

files.download(str(out_path))

# --- Publishing this result is deliberately NOT automated (same
# structural-non-execution pattern as the HF Hub upload guard -- pushing
# results is a reviewed action, not a side effect of running a notebook):
print(
    "\nTo publish this result, on your own machine:\n"
    "  mv ~/Downloads/lineformer-pretrained.json results/lineformer-pretrained.json\n"
    "  rm results/lineformer-pending.json\n"
    "  python scripts/leaderboard/generate.py\n"
    "  git add results/ site/ && git commit -m 'results: LineFormer pretrained baseline (Colab run)' "
    "&& git push origin main"
)